# NaniGPT — Day 1: Setup + First Inference

**Goal of this notebook:** prove Gemma 4 E4B can do (a) caregiving-scenario text reasoning and (b) read a pill photo. Once both work, we're unblocked for Days 2-7.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU** (free tier is fine).


## Step 1 — Install dependencies


In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

## Step 2 — Verify GPU


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 'N/A')

## Step 3 — Load Gemma 4 E4B


In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-e4b-it",   # if errors: try "unsloth/gemma-4-E4B-it" or "unsloth/gemma-4-e2b-it"
    max_seq_length = 8192,
    load_in_4bit = True,
    full_finetuning = False,
)
print('Loaded.')

## Step 4 — Test 1: Caregiver text reasoning

Two non-obvious things to remember about Gemma 4's multimodal processor:
1. `content` must be a **list of typed blocks**, even for plain text. A bare string throws `TypeError: string indices must be integers`.
2. `apply_chat_template` should be called with **`return_dict=True`** so images survive — without it, `pixel_values` get dropped silently and the model only sees text. Then call `model.generate(**inputs, ...)` (note the `**`).

We use the same pattern for both Test 1 (text-only) and Test 2 (image+text) for consistency.


In [ ]:
system_prompt = '''You are NaniGPT, a private AI companion for adult children caring for aging parents with dementia. You speak warmly but briefly. You never give medical advice — you help organize, log, and remind. Output structured JSON when asked, plain language otherwise.'''

user_msg = '''Just got home from work. Mom's pill organizer for Tuesday morning is empty but I'm not sure if she took them or forgot to fill the slot last week. She seems more confused than usual today and didn't want to eat lunch. There's a small bruise on her left forearm I haven't seen before. What should I log right now and what should I follow up on tomorrow?'''

messages = [
    {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
    {"role": "user",   "content": [{"type": "text", "text": user_msg}]},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## Step 5 — Test 2: Pill photo (multimodal vision)

Upload a pill organizer photo from your workspace folder (`Gemma Kaggle Hack/nanigpt/data/test_inputs/pill_organizer_test1.jpg`).


In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()
filename = list(uploaded.keys())[0]
img = Image.open(filename).convert("RGB")
img.thumbnail((768, 768))
img

In [ ]:
messages = [
    {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
    {"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": "This is mom's weekly pill organizer photographed at 6pm Tuesday. For each slot you can see, tell me: day label, time-of-day label (AM/PM), and whether it has pills or is empty. Output JSON."},
    ]},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

_ = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.4,
    top_p=0.9,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## Day 1 done — what comes next

If both tests passed:
- ✅ Gemma 4 E4B is viable for NaniGPT
- → Day 2-3: build a 5-photo pill recognition test set, 5-photo bruise/fall test set, 3-clip voice journal test set, and measure accuracy

If Test 2 was wrong (model hallucinated days/pills):
- Drop slot-by-slot recognition; keep only "did anything change vs yesterday's photo?" comparison detection
- Or: fine-tune E4B on a small pill-image dataset (Day 4-7, Unsloth $10K prize fit)

**Save successful prompt+response pairs to `data/test_outputs/` for the writeup and video.**
